# CHIMERA-Bench Demo

This notebook walks through the CHIMERA-Bench dataset: loading complexes, inspecting features, visualizing structures, and running evaluation.

In [ ]:
import os
import torch
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# Set CHIMERA_DATA_ROOT env var or edit this path
DATA_ROOT = Path(os.environ.get("CHIMERA_DATA_ROOT", "./data/chimera-bench-v1.0"))

print(f"Data root: {DATA_ROOT}")
print(f"Contents: {sorted([p.name for p in DATA_ROOT.iterdir()])}")

## 1. Dataset Metadata

The metadata CSV has 2,922 complexes with 32 columns covering PDB info, species, resolution, chain assignments, and more.

In [ ]:
summary = pd.read_csv(DATA_ROOT / "metadata" / "final_summary.csv")
print(f"Total complexes: {len(summary)}")
print(f"Columns: {list(summary.columns)}")
summary.head()

In [ ]:
# Dataset statistics
print(f"Resolution: mean={summary['resolution'].mean():.2f} A, median={summary['resolution'].median():.2f} A")
print(f"Methods: {summary['method'].value_counts().to_dict()}")
print(f"Antigen types: {summary['antigen_type'].value_counts().to_dict()}")
print(f"Heavy species: {summary['heavy_species'].value_counts().head(5).to_dict()}")
print(f"SKEMPI-linked: {summary['has_skempi'].sum()}")

## 2. Splits

Three biologically motivated splits test generalization along different axes.

In [ ]:
splits = {}
for name in ["epitope_group", "antigen_fold", "temporal"]:
    with open(DATA_ROOT / "splits" / f"{name}.json") as f:
        splits[name] = json.load(f)
    print(f"{name}: train={len(splits[name]['train'])}, val={len(splits[name]['val'])}, test={len(splits[name]['test'])}")

## 3. Loading a Complex

Each complex is a `.pt` file containing sequences, atom coordinates, annotations, numbering, and surface features.

In [ ]:
# Load a test complex from the epitope_group split
cid = splits["epitope_group"]["test"][0]
feat = torch.load(DATA_ROOT / "complex_features" / f"{cid}.pt", weights_only=False)

print(f"Complex ID: {feat['complex_id']}")
print(f"Heavy chain: {len(feat['heavy_sequence'])} residues")
print(f"Light chain: {len(feat['light_sequence'])} residues")
print(f"Antigen:     {len(feat['antigen_sequence'])} residues")
print(f"\nAll keys: {feat.keys()}")

In [ ]:
# Inspect shapes of coordinate arrays
for key in sorted(feat.keys()):
    val = feat[key]
    if isinstance(val, np.ndarray):
        print(f"{key:40s} ndarray  shape={str(val.shape):20s} dtype={val.dtype}")
    elif isinstance(val, str):
        print(f"{key:40s} str      len={len(val)}")
    elif isinstance(val, (list, tuple)):
        print(f"{key:40s} {type(val).__name__:8s} len={len(val)}")
    elif isinstance(val, dict):
        print(f"{key:40s} dict     keys={list(val.keys())}")
    else:
        print(f"{key:40s} {type(val).__name__:8s} = {val}")

## 4. Sequences and Numbering

In [ ]:
# Heavy chain sequence
print("Heavy chain sequence:")
print(feat['heavy_sequence'])

# IMGT numbering
imgt_h = feat['numbering']['imgt']['heavy']
print(f"\nIMGT numbering (first 10 residues): {imgt_h[:10]}")

# CDR masks: -1=framework, 0=CDR1, 1=CDR2, 2=CDR3
cdr_h = feat['cdr_masks']['imgt']['heavy']
for cdr_idx, cdr_name in enumerate(['CDR-H1', 'CDR-H2', 'CDR-H3']):
    positions = [i for i, x in enumerate(cdr_h) if x == cdr_idx]
    if positions:
        seq = ''.join(feat['heavy_sequence'][p] for p in positions)
        print(f"{cdr_name}: positions {positions[0]}-{positions[-1]}, sequence = {seq}")

In [ ]:
# Same for light chain CDRs (mask values: 3=CDR-L1, 4=CDR-L2, 5=CDR-L3)
cdr_l = feat['cdr_masks']['imgt']['light']
for cdr_idx, cdr_name in zip([3, 4, 5], ['CDR-L1', 'CDR-L2', 'CDR-L3']):
    positions = [i for i, x in enumerate(cdr_l) if x == cdr_idx]
    if positions:
        seq = ''.join(feat['light_sequence'][p] for p in positions)
        print(f"{cdr_name}: positions {positions[0]}-{positions[-1]}, sequence = {seq}")

## 5. Epitope and Paratope

In [ ]:
print(f"Epitope residues ({len(feat['epitope_residues'])}):")
for chain, resid, resname in feat['epitope_residues'][:10]:
    print(f"  Chain {chain}, Res {resid}, {resname}")
if len(feat['epitope_residues']) > 10:
    print(f"  ... and {len(feat['epitope_residues']) - 10} more")

print(f"\nParatope residues ({len(feat['paratope_residues'])}):")
for chain, resid, resname in feat['paratope_residues'][:10]:
    print(f"  Chain {chain}, Res {resid}, {resname}")

print(f"\nContact pairs ({len(feat['contact_pairs'])}):")
for cp in feat['contact_pairs'][:5]:
    ab_chain, ab_resid, ab_resname, ag_chain, ag_resid, ag_resname, dist = cp
    print(f"  Ab {ab_chain}:{ab_resname}{ab_resid} -- Ag {ag_chain}:{ag_resname}{ag_resid}  ({dist:.2f} A)")

## 6. 3D Structure Visualization

Plot the CA coordinates of a complex, colored by chain, with epitope and paratope residues highlighted.

In [ ]:
fig = plt.figure(figsize=(12, 5))

# --- Left panel: full complex ---
ax1 = fig.add_subplot(121, projection='3d')
h_ca = feat['heavy_ca_coords']
l_ca = feat['light_ca_coords']
a_ca = feat['antigen_ca_coords']

ax1.scatter(*h_ca.T, s=8, alpha=0.5, label='Heavy', c='steelblue')
ax1.scatter(*l_ca.T, s=8, alpha=0.5, label='Light', c='coral')
ax1.scatter(*a_ca.T, s=8, alpha=0.5, label='Antigen', c='gray')

# Highlight CDR-H3
cdr_h3_pos = [i for i, x in enumerate(feat['cdr_masks']['imgt']['heavy']) if x == 2]
if cdr_h3_pos:
    h3_ca = h_ca[cdr_h3_pos]
    ax1.scatter(*h3_ca.T, s=40, c='red', marker='o', label='CDR-H3', zorder=5)

ax1.set_title(f"{feat['complex_id']}")
ax1.legend(fontsize=8)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# --- Right panel: interface zoom ---
ax2 = fig.add_subplot(122, projection='3d')

# Get epitope CA coords
epi_indices = []
for chain, resid, resname in feat['epitope_residues']:
    # Find index in antigen sequence (approximate by position)
    epi_indices.append(resid if isinstance(resid, int) else int(resid))

# Plot paratope and epitope
para_h = [i for i, x in enumerate(feat['cdr_masks']['imgt']['heavy']) if x >= 0]
para_l = [i for i, x in enumerate(feat['cdr_masks']['imgt']['light']) if x >= 0]

if para_h:
    ax2.scatter(*h_ca[para_h].T, s=20, c='steelblue', alpha=0.7, label='H CDRs')
if para_l:
    ax2.scatter(*l_ca[para_l].T, s=20, c='coral', alpha=0.7, label='L CDRs')
ax2.scatter(*a_ca.T, s=8, c='lightgray', alpha=0.3, label='Antigen')

ax2.set_title('CDR regions + antigen')
ax2.legend(fontsize=8)
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

plt.tight_layout()
plt.show()

## 7. Surface Features

In [ ]:
print(f"Antigen surface points: {feat['ag_surface_points'].shape}")
print(f"Antigen surface normals: {feat['ag_surface_normals'].shape}")
print(f"Antigen surface curvatures: {feat['ag_surface_curvatures'].shape} (mean, Gaussian)")
print(f"Antigen surface chemical feats: {feat['ag_surface_chemical_feats'].shape}")
print(f"  (hydropathy, charge, H-bond donor, H-bond acceptor, aromaticity, polarity)")

# Visualize curvature on surface
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
pts = feat['ag_surface_points']
curv = feat['ag_surface_curvatures']

for i, (ax, name) in enumerate(zip(axes, ['Mean curvature', 'Gaussian curvature'])):
    sc = ax.scatter(pts[:, 0], pts[:, 1], c=curv[:, i], cmap='RdBu_r', s=20)
    ax.set_title(f'Antigen surface: {name}')
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    plt.colorbar(sc, ax=ax)

plt.tight_layout()
plt.show()

## 8. Working with PDB Structures

In [ ]:
# PDB files are included for direct structural analysis
pdb_dir = DATA_ROOT / "structures"
pdb_files = list(pdb_dir.glob("*.pdb"))
print(f"PDB structures available: {len(pdb_files)}")
print(f"Example: {pdb_files[0].name}")

# Read a PDB file
with open(pdb_files[0]) as f:
    lines = f.readlines()
print(f"Lines in PDB: {len(lines)}")
print(f"First ATOM line: {next(l for l in lines if l.startswith('ATOM')).strip()}")